In [1]:
###############################################################################
# 2) Stir for 15 s, stop, then take an OD snapshot
###############################################################################
import time
import requests
from urllib.parse import quote

BASE = "http://leadere1.local"
UNIT = "leaderE1"
EXP  = "overnight_od_via_script"

def run_stirring(rpm=500):
    url = f"{BASE}/api/workers/{UNIT}/jobs/run/job_name/stirring/experiments/{quote(EXP)}"
    resp = requests.patch(url, json={"options": {"target_rpm": rpm}},
                          headers={"Content-Type": "application/json"})
    print("START:", resp.status_code, resp.text)
    return resp

def update_stirring(rpm):
    url = f"{BASE}/api/workers/{UNIT}/jobs/update/job_name/stirring/experiments/{quote(EXP)}"
    resp = requests.patch(url, json={"settings": {"target_rpm": rpm}},
                          headers={"Content-Type": "application/json"})
    print("UPDATE:", resp.status_code, resp.text)
    return resp

def list_running():
    url = f"{BASE}/api/workers/{UNIT}/jobs/running"
    r = requests.get(url)
    try:
        txt = r.text
    except Exception:
        txt = ""
    print("RUNNING:", r.status_code, txt)
    return r

def wait_until_stopped(job_name="stirring", timeout_s=15, poll_s=0.5):
    """Poll /jobs/running until job_name disappears or timeout."""
    deadline = time.time() + timeout_s
    url = f"{BASE}/api/workers/{UNIT}/jobs/running"
    while time.time() < deadline:
        r = requests.get(url)
        if r.ok and (job_name not in r.text):
            return True
        time.sleep(poll_s)
    return False

def stop_stirring():
    # Preferred “stop this specific job” endpoint
    stop_specific = f"{BASE}/api/workers/{UNIT}/jobs/stop/job_name/stirring/experiments/{quote(EXP)}"
    resp = requests.patch(stop_specific, headers={"Content-Type": "application/json"})
    print("STOP specific:", resp.status_code, getattr(resp, "text", ""))

    if wait_until_stopped(job_name="stirring"):
        print("Confirmed: stirring stopped.")
        return True

    # Fallback #1: stop ALL jobs for this unit in this experiment
    stop_all = f"{BASE}/api/workers/{UNIT}/jobs/stop/experiments/{quote(EXP)}"
    resp2 = requests.patch(stop_all, headers={"Content-Type": "application/json"})
    print("STOP all-in-exp:", resp2.status_code, getattr(resp2, "text", ""))

    if wait_until_stopped(job_name="stirring"):
        print("Confirmed after stop-all: stirring stopped.")
        return True

    # Fallback #2: unit_api stop (query params)
    unit_api_stop = f"{BASE}/unit_api/jobs/stop?job_name=stirring&experiment={quote(EXP)}"
    resp3 = requests.patch(unit_api_stop, headers={"Content-Type": "application/json"})
    print("STOP unit_api:", resp3.status_code, getattr(resp3, "text", ""))

    ok = wait_until_stopped(job_name="stirring")
    print("Final stop status:", "stopped" if ok else "still running")
    return ok

def run_od_snapshot(timeout_s=20):
    """Start od_reading with snapshot=True so it takes one reading and exits."""
    run_url = f"{BASE}/api/workers/{UNIT}/jobs/run/job_name/od_reading/experiments/{quote(EXP)}"
    payload = {
        "options": {},
        "args": ["--snapshot"]}
    resp = requests.patch(run_url, json=payload, headers={"Content-Type": "application/json"})
    print("OD snapshot run:", resp.status_code, getattr(resp, "text", ""))

    # The job should start and exit quickly; wait until it's no longer listed
    done = wait_until_stopped(job_name="od_reading", timeout_s=timeout_s, poll_s=0.5)
    print("OD snapshot status:", "completed" if done else "still running / timed out")
    return resp, done


def cycle_it():
    # 1) start stirring
    run_stirring(600)
    # 2) let it run for SS seconds
    time.sleep(60)
    # 3) stop stirring (with verification + fallbacks)
    stop_stirring()
    # 4) trigger single-shot OD reading
    time.sleep(2)
    run_od_snapshot()

if __name__ == "__main__":
    try:
        while True:            # run until interrupted
            cycle_it()
            time.sleep(2)      # optional pause between cycles
    except KeyboardInterrupt:
        print("\nProgram stopped by user (Ctrl-C).")
        # stop_stirring()        # make sure stirring is off at exit



START: 202 {"unit":"leaderE1","task_id":"fed39cb6-6561-4037-99b0-b48170aa560a","result_url_path":"/unit_api/task_results/fed39cb6-6561-4037-99b0-b48170aa560a"}
STOP specific: 202 {"status":"success"}
Confirmed: stirring stopped.
OD snapshot run: 202 {"unit":"leaderE1","task_id":"cbb920f7-1736-41b6-a094-5baf94eee65d","result_url_path":"/unit_api/task_results/cbb920f7-1736-41b6-a094-5baf94eee65d"}
OD snapshot status: completed
START: 202 {"unit":"leaderE1","task_id":"2b9cb619-56a8-4aa2-89b1-69b5b807752f","result_url_path":"/unit_api/task_results/2b9cb619-56a8-4aa2-89b1-69b5b807752f"}
STOP specific: 202 {"status":"success"}
Confirmed: stirring stopped.
OD snapshot run: 202 {"unit":"leaderE1","task_id":"2394f68c-8dc6-4634-99b1-6bc65d4e9588","result_url_path":"/unit_api/task_results/2394f68c-8dc6-4634-99b1-6bc65d4e9588"}
OD snapshot status: completed
START: 202 {"unit":"leaderE1","task_id":"9f9e0689-275e-4012-89e7-524a4410b59e","result_url_path":"/unit_api/task_results/9f9e0689-275e-4012-89